## 0. Imports & Load


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
OUTPUT_DIR = Path("..") / "data_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
master_bookings_df = pd.read_parquet(OUTPUT_DIR / "bookings_cleaned.parquet")


## 1. Load & Basic Clean


In [ ]:
print(f"Loaded: {len(master_bookings_df):,} rows")


In [ ]:
before_rows = len(master_bookings_df)
master_bookings_df.drop_duplicates(inplace=True)
print(f"Rows before: {before_rows:,}")
print(f"Rows after:  {len(master_bookings_df):,}")
print(f"Duplicates removed: {before_rows - len(master_bookings_df):,}")


In [ ]:
outlier_ids = [4503, 4502, 933, 837]
master_bookings = master_bookings_df[~master_bookings_df['restaurant_id'].isin(outlier_ids)]
print(f"After removing outlier restaurants: {len(master_bookings):,} rows")


In [ ]:
# master_bookings already creates booking_date in new_eda — replicate here
master_bookings['booking_date'] = pd.to_datetime(master_bookings['date'], errors='coerce')
master_bookings['day_of_week_index'] = master_bookings['booking_date'].dt.weekday
master_bookings['day_of_week']       = master_bookings['booking_date'].dt.day_name()


## 2. Revenue Normalisation

`master_bookings` stores prices in **cents** (THB). Dividing by 100 gives THB. Using `package_price` as the primary revenue signal (equivalent to `revenue_thb` in the old pipeline).


In [ ]:
# Prices are stored in cents — convert to THB
# package_price is the per-booking package value; revenue is the net collected amount
# Use package_price as revenue_thb (consistent with old pipeline's revenue_thb)
master_bookings['gmv'] = pd.to_numeric(master_bookings['package_price'], errors='coerce').fillna(0) / 100

# total_guests: master_bookings uses party_size directly
if 'total_guests' not in master_bookings.columns:
    master_bookings['total_guests'] = pd.to_numeric(master_bookings['party_size'], errors='coerce').fillna(0)

print(master_bookings[['id', 'restaurant_name', 'booking_date', 'party_size', 'total_guests', 'gmv']].head())


## 3. Valid Booking Filter


In [ ]:
# # Apply the same validity filter used in the ClickHouse query
# valid_bookings_df = master_bookings[
#     (master_bookings['active']              == True) &
#     (master_bookings['no_show']             == False) &
#     (master_bookings['is_temporary']        == False) &
#     (master_bookings['for_locking_system']  == False) &
#     (master_bookings['channel']             != 5)
# ].copy()

# print(f"Valid bookings: {len(valid_bookings_df):,} rows")

valid_bookings_df = master_bookings.copy()


In [ ]:
# If old/new reservation ID mapping has been merged, remove superseded rows
# (rows whose id appears as another row's old_reservation_id)
if 'old_reservation_id' in valid_bookings_df.columns:
    old_rids = valid_bookings_df['old_reservation_id'].dropna().astype(int)
    before = len(valid_bookings_df)
    valid_bookings_df = valid_bookings_df[~valid_bookings_df['id'].isin(old_rids)].reset_index(drop=True)
    print(f"Removed {before - len(valid_bookings_df):,} superseded rows via old_reservation_id")
else:
    print("old_reservation_id column not found — skipping superseded-row removal")


In [ ]:
valid_bookings_df.to_parquet(OUTPUT_DIR / "valid_bookings_for_marketing.parquet", index=False)


## 4. Monthly Aggregation


In [ ]:
valid_bookings_df['year_month'] = valid_bookings_df['booking_date'].dt.to_period('M').dt.to_timestamp()

# Filter out future bookings beyond Jan 2026
cutoff_month = pd.Timestamp('2026-01-31')
valid_bookings_df = valid_bookings_df[valid_bookings_df['year_month'] <= cutoff_month].copy()

restaurants_agg = (
    valid_bookings_df
    .groupby(['restaurant_id', 'restaurant_name', 'year_month'], as_index=False)
    .agg(
        monthly_bookings        =('id',            'count'),
        monthly_gmv         =('gmv',   'sum'),
        avg_gmv_per_booking =('gmv',   'mean'),
        avg_guests              =('total_guests',  'mean'),
        active_days             =('booking_date',  lambda x: x.dt.date.nunique()),
    )
)

# Rename restaurant_name -> name to match downstream pipeline
restaurants_agg = restaurants_agg.rename(columns={'restaurant_name': 'name'})

restaurants_agg['monthly_bookings'] = restaurants_agg['monthly_bookings'].fillna(0).astype(int)
restaurants_agg['monthly_gmv']  = restaurants_agg['monthly_gmv'].fillna(0.0)
restaurants_agg['avg_gmv_per_booking'] = (
    restaurants_agg['avg_gmv_per_booking']
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

restaurants_agg = restaurants_agg.sort_values(['name', 'year_month']).reset_index(drop=True)
print(f"Aggregated shape: {restaurants_agg.shape}")
print(f"Columns: {restaurants_agg.columns.tolist()}")
restaurants_agg.head()


## 5. EDA — Booking Volume Heatmap (Top 20)


In [ ]:
start_date = cutoff_month - pd.DateOffset(months=12)

last_12_months_df = restaurants_agg[
    (restaurants_agg['year_month'] >= start_date) &
    (restaurants_agg['year_month'] <= cutoff_month)
].copy()

top_20_per_month = (
    last_12_months_df.sort_values(['year_month', 'monthly_bookings'], ascending=[True, False])
    .groupby('year_month')
    .head(20)
)
top_restaurants = top_20_per_month['name'].unique()

heatmap_data = last_12_months_df[last_12_months_df['name'].isin(top_restaurants)].pivot(
    index='name', columns='year_month', values='monthly_bookings'
)

plt.figure(figsize=(14, 8))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlGnBu', linewidths=.5)
plt.title('Booking Volume Trends: Top 20 Performing Restaurants', fontsize=16)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Active Restaurant Filter


In [ ]:
analysis_end_date = restaurants_agg['year_month'].max()
cutoff_date       = analysis_end_date - pd.DateOffset(months=12)

print(f"Analysis End Date : {analysis_end_date.date()}")
print(f"Activity Cutoff   : {cutoff_date.date()}")
print(f"Data starts from  : {restaurants_agg['year_month'].min().date()}")

latest_activity    = restaurants_agg.groupby('name')['year_month'].max()
active_restaurants = latest_activity[latest_activity >= cutoff_date].index

# Keep FULL history for YoY calculation — prior-year rows needed for pct_change(periods=12)
restaurants_agg_active = restaurants_agg[
    restaurants_agg['name'].isin(active_restaurants)
].copy()

restaurants_agg_active['in_analysis_window'] = (
    restaurants_agg_active['year_month'] >= cutoff_date
)

original_count = restaurants_agg['name'].nunique()
new_count      = restaurants_agg_active['name'].nunique()

print(f"\n--- Filter Summary ---")
print(f"Total Restaurants        : {original_count:,}")
print(f"Active (last 12m)        : {new_count:,}")
print(f"Dropped (churned)        : {original_count - new_count:,}")
print(f"Total rows (full history): {len(restaurants_agg_active):,}")
print(f"Rows in analysis window  : {restaurants_agg_active['in_analysis_window'].sum():,}")
print(f"Prior-year rows retained : {(~restaurants_agg_active['in_analysis_window']).sum():,}  ← needed for YoY")


## 7. Growth Signals — MoM & YoY


In [ ]:
MAX_GROWTH_CAP = 2.0   # +200% cap
GROWTH_FLOOR   = -1.0  # -100% floor
ROLL           = 3     # 3-month rolling window
YOY_MIN        = 2     # require >=2 of 3 months valid YoY data

# ── Step 1: Month-over-month growth ──────────────────────────────────────────
restaurants_agg_active['booking_growth_mom'] = (
    restaurants_agg_active.groupby('name')['monthly_bookings']
    .pct_change()
    .replace([np.inf, -np.inf], [MAX_GROWTH_CAP, GROWTH_FLOOR])
    .fillna(0)
    .clip(upper=MAX_GROWTH_CAP, lower=GROWTH_FLOOR)
)
restaurants_agg_active['gmv_growth_mom'] = (
    restaurants_agg_active.groupby('name')['monthly_gmv']
    .pct_change()
    .replace([np.inf, -np.inf], [MAX_GROWTH_CAP, GROWTH_FLOOR])
    .fillna(0)
    .clip(upper=MAX_GROWTH_CAP, lower=GROWTH_FLOOR)
)

# ── Step 2: Year-over-year growth — merge-based (not positional) ─────────────
prior_year = restaurants_agg_active[
    ['name', 'year_month', 'monthly_bookings', 'monthly_gmv']
].copy()
prior_year['year_month'] = prior_year['year_month'] + pd.DateOffset(years=1)
prior_year = prior_year.rename(columns={
    'monthly_bookings': 'bookings_prior_year',
    'monthly_gmv' : 'gmv_prior_year',
})
restaurants_agg_active = restaurants_agg_active.merge(
    prior_year[['name', 'year_month', 'bookings_prior_year', 'gmv_prior_year']],
    on=['name', 'year_month'],
    how='left',
)

restaurants_agg_active['booking_growth_yoy'] = (
    (restaurants_agg_active['monthly_bookings'] - restaurants_agg_active['bookings_prior_year'])
    / restaurants_agg_active['bookings_prior_year'].replace(0, np.nan)
).replace([np.inf, -np.inf], [MAX_GROWTH_CAP, GROWTH_FLOOR]).clip(upper=MAX_GROWTH_CAP, lower=GROWTH_FLOOR)

restaurants_agg_active['gmv_growth_yoy'] = (
    (restaurants_agg_active['monthly_gmv'] - restaurants_agg_active['gmv_prior_year'])
    / restaurants_agg_active['gmv_prior_year'].replace(0, np.nan)
).replace([np.inf, -np.inf], [MAX_GROWTH_CAP, GROWTH_FLOOR]).clip(upper=MAX_GROWTH_CAP, lower=GROWTH_FLOOR)

# ── Step 3: History flag ──────────────────────────────────────────────────────
months_of_history = (
    restaurants_agg_active.groupby('name')['year_month']
    .nunique()
    .rename('months_of_history')
)
restaurants_agg_active = restaurants_agg_active.merge(months_of_history, on='name', how='left')
restaurants_agg_active['has_full_year'] = restaurants_agg_active['months_of_history'] >= 12

print(f"Restaurants with >=12 months (YoY candidate): {restaurants_agg_active[restaurants_agg_active['has_full_year']]['name'].nunique():,}")
print(f"Restaurants with  <12 months (MoM only)     : {restaurants_agg_active[~restaurants_agg_active['has_full_year']]['name'].nunique():,}")

# ── Step 4: MoM 3-month rolling ──────────────────────────────────────────────
restaurants_agg_active['booking_growth_mom_rolling'] = (
    restaurants_agg_active.groupby('name')['booking_growth_mom']
    .rolling(ROLL, min_periods=ROLL)
    .mean()
    .reset_index(level=0, drop=True)
    .fillna(0.0)
)
restaurants_agg_active['gmv_growth_mom_rolling'] = (
    restaurants_agg_active.groupby('name')['gmv_growth_mom']
    .rolling(ROLL, min_periods=ROLL)
    .mean()
    .reset_index(level=0, drop=True)
    .fillna(0.0)
)

# ── Step 5: YoY 3-month rolling (min_periods=YOY_MIN) ────────────────────────
restaurants_agg_active['booking_growth_yoy_rolling'] = (
    restaurants_agg_active.groupby('name')['booking_growth_yoy']
    .rolling(ROLL, min_periods=YOY_MIN)
    .mean()
    .reset_index(level=0, drop=True)
)
restaurants_agg_active['gmv_growth_yoy_rolling'] = (
    restaurants_agg_active.groupby('name')['gmv_growth_yoy']
    .rolling(ROLL, min_periods=YOY_MIN)
    .mean()
    .reset_index(level=0, drop=True)
)

# Nullify YoY rolling for restaurants without a full year
mask_no_yoy = ~restaurants_agg_active['has_full_year']
restaurants_agg_active.loc[mask_no_yoy, 'booking_growth_yoy_rolling'] = np.nan
restaurants_agg_active.loc[mask_no_yoy, 'gmv_growth_yoy_rolling'] = np.nan

# ── Step 6: Signal selection ──────────────────────────────────────────────────
use_yoy = (
    restaurants_agg_active['has_full_year'] &
    restaurants_agg_active['booking_growth_yoy_rolling'].notna()
)
restaurants_agg_active['growth_signal_used'] = np.where(use_yoy, 'YoY', 'MoM')

# Backward-compatible blended rolling columns (used by segmentation + dashboard)
restaurants_agg_active['booking_growth_rolling'] = np.where(
    use_yoy,
    restaurants_agg_active['booking_growth_yoy_rolling'],
    restaurants_agg_active['booking_growth_mom_rolling'],
)
restaurants_agg_active['gmv_growth_rolling'] = np.where(
    use_yoy,
    restaurants_agg_active['gmv_growth_yoy_rolling'],
    restaurants_agg_active['gmv_growth_mom_rolling'],
)

print("\nGrowth signal used (latest snapshot):")
latest_snap = restaurants_agg_active.sort_values('year_month').groupby('name').tail(1)
print(latest_snap['growth_signal_used'].value_counts())


In [ ]:
restaurants_agg_active.columns


## 8. Visualise — YoY vs MoM Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

latest_snap = restaurants_agg_active.sort_values('year_month').groupby('name').tail(1)

axes[0].hist(
    latest_snap[latest_snap['growth_signal_used'] == 'YoY']['booking_growth_rolling'].clip(-1, 2),
    bins=30, alpha=0.6, color='steelblue', label='YoY (seasonality-adjusted)'
)
axes[0].hist(
    latest_snap[latest_snap['growth_signal_used'] == 'MoM']['booking_growth_rolling'].clip(-1, 2),
    bins=30, alpha=0.6, color='salmon', label='MoM (fallback)'
)
axes[0].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[0].set_title('Booking Growth Distribution: YoY vs MoM')
axes[0].set_xlabel('Growth Rate')
axes[0].legend()

signal_counts = latest_snap['growth_signal_used'].value_counts()
axes[1].bar(signal_counts.index, signal_counts.values, color=['steelblue', 'salmon'])
axes[1].set_title('Growth Signal Used per Restaurant')
axes[1].set_ylabel('Number of Restaurants')

plt.tight_layout()
plt.show()


## 9. Performance & Growth Scores


In [ ]:
def min_max(s):
    rng = s.max() - s.min()
    if rng == 0:
        return pd.Series(0.5, index=s.index)
    return (s - s.min()) / rng

restaurants_agg_active['log_bookings'] = np.log1p(restaurants_agg_active['monthly_bookings'])
restaurants_agg_active['log_gmv']      = np.log1p(restaurants_agg_active['monthly_gmv'])

analysis_rows = restaurants_agg_active['in_analysis_window']

# ── Performance score ─────────────────────────────────────────────────────────
restaurants_agg_active['score_perf'] = (
    min_max(restaurants_agg_active['log_bookings']) * 0.5 +
    min_max(restaurants_agg_active['log_gmv'])      * 0.5
)

# ── Raw composite rates (before normalisation) ────────────────────────────────
mom_raw = pd.Series(
    restaurants_agg_active['gmv_growth_mom_rolling'] * 0.5 +
    restaurants_agg_active['booking_growth_mom_rolling'] * 0.5,
    index=restaurants_agg_active.index
)

yoy_raw = pd.Series(
    restaurants_agg_active['gmv_growth_yoy_rolling'].fillna(np.nan) * 0.5 +
    restaurants_agg_active['booking_growth_yoy_rolling'].fillna(np.nan) * 0.5,
    index=restaurants_agg_active.index
)

yoy_valid = (
    restaurants_agg_active['has_full_year'] &
    restaurants_agg_active['booking_growth_yoy_rolling'].notna()
)
yoy_raw = yoy_raw.where(yoy_valid)  # NaN for MoM-only restaurants

# ── Blended raw rate ──────────────────────────────────────────────────────────
blended_raw = np.where(
    yoy_valid,
    yoy_raw * 0.70 + mom_raw * 0.30,
    mom_raw
)
restaurants_agg_active['growth_rate_blended'] = blended_raw

# ── Single normalisation ──────────────────────────────────────────────────────
restaurants_agg_active['score_growth'] = min_max(
    pd.Series(blended_raw, index=restaurants_agg_active.index)
)

# ── Individual scores kept for dashboard transparency ─────────────────────────
restaurants_agg_active['score_growth_mom'] = min_max(mom_raw)

yoy_raw_filled = yoy_raw.copy()
restaurants_agg_active['score_growth_yoy'] = min_max(yoy_raw_filled.fillna(yoy_raw_filled.median()))
restaurants_agg_active.loc[~yoy_valid, 'score_growth_yoy'] = np.nan

# ── Seasonal flag ─────────────────────────────────────────────────────────────
mom_median = mom_raw.median()
yoy_median = yoy_raw.dropna().median()

restaurants_agg_active['is_seasonal'] = (
    yoy_valid &
    (mom_raw >= mom_median) &
    (yoy_raw.fillna(-999) < yoy_median)
)

print(f"Scores computed — analysis window rows: {analysis_rows.sum():,}")
print(restaurants_agg_active[['score_perf', 'score_growth', 'score_growth_mom', 'score_growth_yoy']].describe().round(3))
print(f"\nRestaurants flagged Seasonal: {restaurants_agg_active[restaurants_agg_active['is_seasonal']]['name'].nunique():,}")


## 10. Segmentation


In [ ]:
analysis_window_df = restaurants_agg_active[
    restaurants_agg_active['in_analysis_window']
].copy()

latest = analysis_window_df.sort_values('year_month').groupby('name').tail(1)
print(f"Restaurants in latest snapshot: {latest['name'].nunique():,}")

perf_threshold   = latest['score_perf'].quantile(0.90)
growth_threshold = latest['score_growth'].quantile(0.90)

print(f"Performance Cutoff (90th%): {perf_threshold:.3f}")
print(f"Growth Cutoff (90th%)     : {growth_threshold:.3f}")

def get_segment(row):
    is_big     = row['score_perf']   >= perf_threshold
    is_growing = row['score_growth'] >= growth_threshold
    if is_big and is_growing:     return 'Rising Stars'
    if not is_big and is_growing: return 'Emerging Opportunities'
    if is_big and not is_growing: return 'Established Players'
    return 'Needs Attention'

latest['latest_segment'] = latest.apply(get_segment, axis=1)
print("\nSegment distribution:")
print(latest['latest_segment'].value_counts())


## 11. Acceleration Signals (Delta Columns)


In [ ]:
cols_to_shift = [
    'booking_growth_rolling', 'gmv_growth_rolling',
    'log_bookings', 'log_gmv'
]

for col in cols_to_shift:
    analysis_window_df[f'{col}_prev'] = (
        analysis_window_df.groupby('name')[col]
        .shift(1)
        .fillna(0.0)
    )

analysis_window_df['delta_growth_book'] = (
    analysis_window_df['booking_growth_rolling'] -
    analysis_window_df['booking_growth_rolling_prev']
)
analysis_window_df['delta_growth_rev'] = (
    analysis_window_df['gmv_growth_rolling'] -
    analysis_window_df['gmv_growth_rolling_prev']
)
analysis_window_df['delta_size_book'] = (
    analysis_window_df['log_bookings'] - analysis_window_df['log_bookings_prev']
)
analysis_window_df['delta_size_rev'] = (
    analysis_window_df['log_gmv'] - analysis_window_df['log_gmv_prev']
)

print("Delta columns computed.")
print(analysis_window_df[['delta_growth_book', 'delta_growth_rev']].describe().round(3))


## 12. Visualise — Strategic Matrix & Growth Heatmaps


In [ ]:
highlights_df = analysis_window_df.sort_values('year_month').groupby('name').tail(1)

top_revenue_growth = highlights_df.nlargest(10, 'gmv_growth_rolling')
top_booking_growth = highlights_df.nlargest(10, 'booking_growth_rolling')
top_volume         = highlights_df.nlargest(10, 'monthly_bookings')
highlights = pd.concat([top_revenue_growth, top_booking_growth, top_volume]).drop_duplicates()

plt.figure(figsize=(14, 9))
plt.scatter(
    highlights_df['monthly_bookings'], highlights_df['booking_growth_rolling'],
    c='lightgrey', s=30, alpha=0.4, label='Others'
)
bubble_size = highlights['monthly_gmv'] / highlights['monthly_gmv'].max() * 2000
scatter = plt.scatter(
    highlights['monthly_bookings'], highlights['booking_growth_rolling'],
    s=bubble_size, c=highlights['booking_growth_rolling'],
    cmap='RdYlGn', edgecolors='black', alpha=0.9, zorder=10
)
for _, row in highlights.iterrows():
    signal = row.get('growth_signal_used', '')
    plt.text(row['monthly_bookings'] + 0.05, row['booking_growth_rolling'] + 0.02,
             f"{row['name']} ({signal})", fontsize=9, fontweight='bold', color='black', zorder=11)

plt.axhline(0.2, color='gray', linestyle='--', alpha=0.5)
plt.axvline(highlights_df['monthly_bookings'].median(), color='gray', linestyle='--', alpha=0.5)
plt.title('Strategic Matrix: Volume vs Momentum\n(YoY/MoM signal labelled)', fontsize=14)
plt.xlabel('Volume (Monthly Bookings)')
plt.ylabel('Momentum (Rolling Growth — YoY where available)')
plt.tight_layout()
plt.show()


In [ ]:
# Growth heatmap — top 20 by volume
latest_month_label = restaurants_agg_active['year_month'].max()
latest_growth      = restaurants_agg_active[restaurants_agg_active['year_month'] == latest_month_label].copy()
top_20_names       = latest_growth.nlargest(20, 'monthly_bookings')['name'].tolist()

growth_trend_df = last_12_months_df[last_12_months_df['name'].isin(top_20_names)].merge(
    restaurants_agg_active[['name', 'year_month', 'booking_growth_rolling', 'gmv_growth_rolling']],
    on=['name', 'year_month'], how='left'
)

growth_heatmap = growth_trend_df.pivot(index='name', columns='year_month', values='gmv_growth_rolling')
growth_heatmap.columns = [d.strftime('%b %Y') for d in growth_heatmap.columns]

plt.figure(figsize=(16, 10))
sns.heatmap(growth_heatmap, annot=True, fmt='.1%', cmap='RdYlGn', center=0,
            linewidths=.5, cbar_kws={'label': 'Rolling Revenue Growth %'})
plt.title('Growth Momentum Heatmap: Revenue Growth % (Past 12 Months)', fontsize=18, pad=20)
plt.ylabel('Restaurant Name')
plt.xlabel('Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 13. Save Outputs


In [ ]:
# Save analysis-window dataframe (scored, with delta columns)
# This is what priority_scoring_seasonality.ipynb reads
analysis_window_df.to_parquet(OUTPUT_DIR / "restaurants_agg_performance.parquet")
print("Saved restaurants_agg_performance.parquet")
print(f"Shape      : {analysis_window_df.shape}")
print(f"Date range : {analysis_window_df['year_month'].min().date()} → {analysis_window_df['year_month'].max().date()}")
print(f"Restaurants: {analysis_window_df['name'].nunique():,}")

seasonality_cols = [c for c in analysis_window_df.columns if any(
    x in c for x in ['yoy', 'mom', 'signal', 'full_year', 'months_of_history', 'in_analysis', 'seasonal']
)]
print(f"\nSeasonality columns: {seasonality_cols}")


In [ ]:
# Save latest-month momentum labels
output_path = OUTPUT_DIR / "priority_latest_momentum_labels.parquet"
latest.to_parquet(output_path, index=False)
print(f"Saved priority_latest_momentum_labels.parquet")
print(f"Restaurants: {latest['name'].nunique():,}")
print(f"Segment distribution:")
print(latest['latest_segment'].value_counts())
